# IP extraction notebook for Endgame
This notebook takes endgame specific *.json files and reads them into a breach specific database. Before you run this make sure you update the constants below.

Record format:
```json
[
  {
    "stealer_name": "rhadamanthys",
    "victimIP": "105.245.xx.xx",
    "countryCode": "ZA",
    "last_used": "2025-08-06T01:41:17Z",
    "computerName": "BLA",
    "username": "THE BLA",
    "url": "http://192.168.10.140/",
    "login": "root",
    "password": "****",
    "password_sha1": "dc76e91230006e8f919e0c515c66dbba3982f785",
    "password_ntlm": "329153f560eb123c0e1deea55e88a1e9"
  },
  {
    "stealer_name": "rhadamanthys",
    "victimIP": "105.245.xx.xx",
    "countryCode": "ZA",
    "last_used": "2025-08-06T01:41:17Z",
    "computerName": "BLA",
    "username": "THE BLA",
    "url": "http://192.168.9.140/",
    "login": "root",
    "password": "****",
    "password_sha1": "dc76e91230006e8f919e0c515c66dbba3982f785",
    "password_ntlm": "329153f560eb123c0e1deea55e88a1e9"
  }
]
```

# Set constants

In [ ]:
# change these
CASE="DIVD-2025-00041"
SUB="ips"
# Keep the same
IN_DIR="../IN"
OUT_DIR="../LEAK_DB/"
OUT_DB=f"{OUT_DIR}/{CASE}-{SUB}.sqlite3"

# Import helper functions

In [ ]:
%run ../0.shared_notebooks/0_helper_functions.ipynb

# Import data

In [ ]:
!ls $OUT_DIR


In [ ]:
#!rm $OUT_DIR/*

In [ ]:
!rm log.txt
!rm error_log.txt

# Open DB (or create it)

In [ ]:
if not os.path.exists(OUT_DB) :
    create_ip_db(OUT_DB)
# Open database
conn = sqlite3.connect(OUT_DB)
cur = conn.cursor()

# Import records

In [ ]:
files=sorted(glob(f"{IN_DIR}/*.json"))

In [ ]:
files

In [ ]:
cur.execute("DELETE FROM 'entity'")
count=0
for file in files :
    print(f"File: {file} - ", end="") 
    
    with open(file) as f:
        records = json.load(f)
    print(f"{len(records)} records")
    with open("log.txt", "a") as lfile:
        lfile.write(f"File: {file} - {len(records)} records\n") 
    for row in records:
        slash24 = None
        result = re.search(r'^(\d+\.\d+\.\d+)\.\d+$', row["victimIP"])
        if result :
            slash24 = f"{result.group(1)}.0"
        extradata = row.copy()
        extradata.pop("password",None)
        extradata.pop("password_sha1",None)
        extradata.pop("password_ntlm",None)
        extradata.pop("url",None)
        extradata.pop("login",None)
        try:
            cur.execute("""
                INSERT into 'entity' ( 
                    ip, slash24, username, computername, extra_data
                ) values (
                    ?, ?, ?, ?, ?
                )
            """ , (
                row["victimIP"], slash24, row["username"], 
                row["computerName"], json.dumps(extradata, indent=2)
            ) )
        except Exception as e:
            with open('error_log.txt', 'a') as efile:
                efile.write(f"{datetime.now()} - Error: {e}\n")
            print(f"\n{datetime.now()} - Error: {e}")
        count=count+1
        if count % 10_000 == 0 :
            print(f"{count:,}", end="\r")
        if count % 100_000 == 0 :
            conn.commit()
        #bla()
    with open("log.txt", "a") as lfile:
        lfile.write(f"\n{count:,}") 
conn.commit()
print(f"\n{count:,}")

In [ ]:
conn.commit()